# 26 · The agent loop
Level L3 · Part 4 · core · ~35 min · builds on 22

## Goal
Write a loop that calls the model, runs every tool it asks for, appends the results, and repeats until
the model answers or a step limit is hit. Scout uses it to answer a two-step question (search, then
calculate), first with a fake model and then with Gemini.

## The idea
Lesson 22 made one tool call by hand: call, run, send back, call again. Some questions need more than
one tool, and you cannot know in advance how many. So you stop writing the calls out one by one and put
them in a loop. Each pass is one **step**: call the model; if it asked for tools, run them and append
the results; otherwise it has given its answer, the **end of turn**, and you stop. A **step limit**
stops a model that never finishes. Think of a cook following a recipe: read the next line, do it, read
the next, until the recipe says "serve". The kitchen timer is the step limit.

## Picture
```mermaid
flowchart TD
    Q[question into messages] --> C[call the model]
    C --> T{tool calls?}
    T -- no --> A[end of turn: return the answer]
    T -- yes --> R[run each tool, append results]
    R --> L{step limit hit?}
    L -- no --> C
    L -- yes --> S[stop: step_limit]
```

## Already know this?
This is the first lesson of Part 4. If you can answer all of these, skip to lesson 35 (the checkpoint).
1. Write the agent loop from memory. When does it stop, and what does it return in each case?
2. How do you add a new tool without editing the loop?
3. A tool raises an exception. What should the model see, and why not crash?
4. Name three stopping conditions besides "the model answered".
5. History has outgrown the context window. Name two ways to shrink it, and when you would use a fixed
   workflow (chaining, routing) instead of an agent.

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 22. Without a key the Gemini calls replay from
`../data/llm_cassettes/26_agent-loop.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("26_agent-loop")

LLM mode: replay (26_agent-loop.yaml)


Setup from lesson 22, unchanged: the notes, the client, the pinned model and `search_notes`.

In [2]:
import ast, json, operator, os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

And its schema from lesson 22, also unchanged.

In [3]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}. Use it before answering any question "
                   "about what the notes say.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}},
        "required": ["query"],
    },
}
print("model:", MODEL, "| notes:", len(NOTES))

model: gemini-3.5-flash-lite | notes: 6


### 2. A second tool: a calculator
The question for this lesson is "How many minutes long is a day on Mars?". The notes say "24 hours and
39 minutes", so Scout must search first and then do arithmetic. Models are unreliable at arithmetic, so
the arithmetic goes to a tool. It walks the parsed expression and allows only numbers and `+ - * /`.
It never calls `eval`, which would run any Python the model sent.

In [4]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

CALCULATOR_SCHEMA = {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression with numbers, + - * / and brackets, e.g. '24*60+39'. "
                   "Use it for every calculation instead of doing arithmetic yourself.",
    "parameters": {"type": "object", "required": ["expression"],
                   "properties": {"expression": {"type": "string", "description": "e.g. '(3+4)*2'"}}},
}
TOOLS = {"search_notes": search_notes, "calculator": calculator}
SCHEMAS = [SEARCH_NOTES_SCHEMA, CALCULATOR_SCHEMA]
print(calculator("24*60+39"), calculator("(3+4)*2"))

1479 14


### 3. A scripted fake model
The fake plays a fixed script, one reply per call, and gives each tool call an id. The last entry
repeats if the loop keeps calling. A reply can be a function of the messages, so the final answer is
built from the tool result the loop sent back. Like lesson 22's fake, it checks that every tool result
answers a call it made.

In [5]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

The script for our question: search, then calculate, then answer from the calculator's result.

In [6]:
QUESTION = "How many minutes long is a day on Mars?"
def last_result(messages): return json.loads(messages[-1]["content"])

MARS_DAY_SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "day"}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda messages: {"text": f"A day on Mars lasts about {last_result(messages)} minutes."},
]
fake = ScriptedModel(MARS_DAY_SCRIPT)
print(fake([{"role": "user", "text": QUESTION}], SCHEMAS))

{'role': 'assistant', 'tool_calls': [{'name': 'search_notes', 'args': {'query': 'day'}, 'id': 'call_1'}]}


### 4. The loop
Here it is. Each step is one model call. The model's reply goes into the history first, then one tool
result per tool call, each with that call's id (lesson 22), all of them if there are several (lesson 25).
No tool calls means the end of turn. Running out of steps returns `stop="step_limit"` and no answer.

In [7]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6) -> dict:
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)                      # 1. call the model
        messages.append(reply)                                # its turn goes into the history
        calls = reply.get("tool_calls", [])
        if not calls:                                         # 2. end of turn: it answered
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "messages": messages}
        for call in calls:                                    # 3. run every tool it asked for
            result = tools[call["name"]](**call["args"])
            print(f"step {step}: {call['name']}({call['args']}) -> {json.dumps(result)[:60]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "content": json.dumps(result)})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "messages": messages}  # 4. limit

### 5. Run it with the fake model

In [8]:
run = run_agent(QUESTION, ScriptedModel(MARS_DAY_SCRIPT), TOOLS, SCHEMAS)
print(run["stop"], "after", run["steps"], "steps:", run["answer"])
print("history roles:", [m["role"] for m in run["messages"]])

step 1: search_notes({'query': 'day'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 3
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: answer
end_of_turn after 3 steps: A day on Mars lasts about 1479 minutes.
history roles: ['user', 'assistant', 'tool', 'assistant', 'tool', 'assistant']


Three steps and three model calls: two tool steps, then the answer. The 1479 came from `calculator`,
not from the model. The history alternates assistant and tool turns, every result after its call.

### 6. The step limit
A model can keep asking for tools forever: it searches, does not like the result, searches again. This
fake does exactly that (its one script entry repeats). The loop gives up after `max_steps` and says so.

In [9]:
stubborn = ScriptedModel([{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}])
run = run_agent(QUESTION, stubborn, TOOLS, SCHEMAS, max_steps=3)
print(run["stop"], "| answer:", run["answer"], "| model calls:", stubborn.calls)

step 1: search_notes({'query': 'Pluto'}) -> []
step 2: search_notes({'query': 'Pluto'}) -> []
step 3: search_notes({'query': 'Pluto'}) -> []
step_limit | answer: None | model calls: 3


`stop` tells the caller *why* there is no answer. Lesson 30 adds time limits and cancelling.

### 7. Gemini behind the same interface
The loop only needs `model(messages, schemas) -> reply`. So we wrap Gemini in a class with that shape
and leave the loop alone. First, turning our messages into Gemini `contents`, using lesson 22's rules:
the model's turn goes back unchanged (it carries the thought signature, kept under `"raw"`), and tool
results become `function_response` parts in a `user` turn, all results of one step together (lesson 25).

In [10]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                        # unchanged: keeps the thought signature
        else:                                                # a tool result
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response={"result": json.loads(m["content"])}))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)              # same step: same user turn
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

Now the wrapper. It declares the tools, calls `generate_content`, and translates the reply back into
our format. Remember from lesson 22: Gemini's finish reason stays `STOP` for tool calls, so we decide
by checking `resp.function_calls`.

In [11]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Do every calculation with the calculator tool, never in your head. Answer in one sentence.")

class GeminiModel:
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=t["name"], description=t["description"],
                                           parameters_json_schema=t["parameters"]) for t in tools]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=[types.Tool(function_declarations=decls)],
                                          automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

### 8. The same loop with Gemini
Same question, same tools, same `run_agent`. Only the model changed.

In [12]:
run = run_agent(QUESTION, GeminiModel(), TOOLS, SCHEMAS)
print(run["stop"], "after", run["steps"], "steps:", run["answer"])

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Dei
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: answer
end_of_turn after 3 steps: A day on Mars is 1479 minutes long.


Gemini chose the search query ("Mars") and the expression itself, and the loop ran them. Here is the
whole history in Gemini's shape: the question, a model turn and a user turn per tool step, and the
final text answer.

In [13]:
for c in to_contents(run["messages"]):
    kinds = [k for p in c.parts for k in ("text", "function_call", "function_response") if getattr(p, k)]
    print(f"{c.role:5} {kinds}")

user  ['text']
model ['function_call']
user  ['function_response']
model ['function_call']
user  ['function_response']
model ['text']


## At work
- **The loop is unit-tested with a scripted fake model**, like `ScriptedModel`: step limit, end of turn,
  several tool calls in one step. These tests need no key and never flake. → lesson 46
- **Runaway loops cost real money.** Production agents cap steps, tokens and cost per run, and report
  *why* a run stopped, as `stop` does here. → lessons 30 and 54
- **Tools fail.** Real loops catch a tool's exception and send the error back as the tool result, so
  the model can retry (Anthropic notes Claude retries an invalid tool call 2 or 3 times). → lesson 28
- **Every step is logged:** model, tokens, stop reason, tool name, arguments and result size, so a bad
  answer can be traced to the step that caused it. → lesson 49

## What just happened
- We added a second tool, `calculator`, that evaluates arithmetic safely, without `eval`.
- `run_agent` repeats one step: call the model, append its turn, run every tool call and append each
  result with its id.
- A reply with no tool calls is the **end of turn**: the loop returns the answer with `stop="end_of_turn"`.
- A model that never stops hits the **step limit**: the loop returns `stop="step_limit"` and no answer.
- `GeminiModel` has the same call shape as the fake, so the same loop answered with Gemini: search,
  calculate, answer.

## Common mistakes
**1. `while True` with no step limit.** It works while the model behaves. With the stubborn model it
never ends: our fake gives up after 50 calls, a real model would keep billing you.

In [14]:
def run_agent_forever(question, model, tools, schemas):
    messages = [{"role": "user", "text": question}]
    while True:
        reply = model(messages, schemas)
        messages.append(reply)
        if not reply.get("tool_calls"):
            return reply["text"]
        for call in reply["tool_calls"]:
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "content": json.dumps(tools[call["name"]](**call["args"]))})

run_agent_forever(QUESTION, ScriptedModel(stubborn.script), TOOLS, SCHEMAS)

RuntimeError: model called 50 times: the loop never stopped

Fix: a `for` over `range(max_steps)`, as `run_agent` does. It always ends.

In [15]:
print(run_agent(QUESTION, ScriptedModel(stubborn.script), TOOLS, SCHEMAS, max_steps=2)["stop"])

step 1: search_notes({'query': 'Pluto'}) -> []
step 2: search_notes({'query': 'Pluto'}) -> []
step_limit


**2. Appending the tool results but not the model's turn.** The next call then holds results for calls
the history never made. Our fake rejects it. Gemini cannot work without it either: lesson 22 showed
it needs the model's turn with its thought signature.

In [16]:
model = ScriptedModel(MARS_DAY_SCRIPT)
messages = [{"role": "user", "text": QUESTION}]
reply = model(messages, SCHEMAS)                                   # forgot: messages.append(reply)
call = reply["tool_calls"][0]
messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                 "content": json.dumps(search_notes(**call["args"]))})
model(messages, SCHEMAS)

ValueError: tool result for unknown call id 'call_1'

Fix: append the reply before its results, as `run_agent` does on the line after the model call.

In [17]:
messages.insert(1, reply)
print([m["role"] for m in messages], "->", model(messages, SCHEMAS)["tool_calls"][0]["name"])

['user', 'assistant', 'tool'] -> calculator


## Try it
Ask Scout "How many moons does Mars have, times 3?" with a fake model. Write a script that searches for
"moons", calls the calculator with "2*3", and answers from the calculator's result. Run it through
`run_agent` and check the answer contains 6. Then run it again with `max_steps=2`: what does it return?

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **agent loop** — call the model; if it asks for tools, run them and add the results to the history; repeat until it answers or a limit is hit.
- **step** — one pass of the agent loop: one model call plus the tools it asked for.
- **end of turn** — a model reply with no tool calls: the final answer, and the normal way the loop ends.
- **step limit** — the maximum number of steps (`max_steps`) before the loop stops without an answer and reports `stop="step_limit"`.